## Цикл событий

### 1) Системные вызовы и создание задач
В этом блоке задаётся базовый интерфейс `SystemCall`. Любой объект, который возвращается из корутины через `yield` и наследуется от `SystemCall`, перехватывается планировщиком и обрабатывается отдельно.

`NewTask` — конкретный системный вызов: он регистрирует новую корутину в `Scheduler`, получает её `tid` и возвращает управление родительской задаче.

In [2]:
from typing import Generator
from __future__ import annotations


class SystemCall:
    def handle(self, sched: Scheduler, task: Task):
        """
        Базовый класс для системных вызовов.
        Системные вызовы — это специальные объекты, которые задачи могут "вызывать" (yield),
        чтобы взаимодействовать с планировщиком и операционной системой.
        Метод handle() будет переопределён в подклассах для реализации конкретного поведения системного вызова.
        """
        pass


class NewTask(SystemCall):
    def __init__(self, target: Generator):
        self.target = target

    def handle(self, sched: Scheduler, task: Task):
        """
        Этот системный вызов позволяет создать новую задачу из генератора (корутины)."""
        tid = sched.add_task(self.target)
        task.sendval = tid
        sched.schedule(task)

### 2) Объект Task и выполнение корутин
`Task` хранит состояние одной корутины: идентификатор, текущее значение для `send`, активный генератор и стек вложенных генераторов.

Метод `run()` исполняет задачу до ближайшей точки переключения:
- если корутина вернула `SystemCall`, выполнение передаётся планировщику;
- если корутина вернула другой генератор, он кладётся на стек как вложенный вызов;
- при завершении генератора (`StopIteration`) управление возвращается на предыдущий уровень стека.

In [3]:
import types
from typing import Generator


class Task:
    # Счётчик для присвоения уникального id каждой задаче
    task_id = 0

    def __init__(self, target: Generator):
        Task.task_id += 1
        self.tid = Task.task_id      # Уникальный идентификатор задачи
        self.target = target         # Текущий активный генератор (корутина)
        self.sendval = None          # Значение, которое будет отправлено в генератор через send()
        self.stack = []              # Стек вложенных генераторов (для поддержки yield из yield)

    def run(self):
        """
        Выполняет задачу до следующего переключения управления.
        Поддерживает вложенные генераторы (корутины),
        реализуя простейший механизм кооперативного планировщика.
        """
        while True:
            try:
                # Передаём значение в текущий генератор
                # и получаем то, что он "вернул" через yield
                result = self.target.send(self.sendval)

                # Если генератор вернул объект SystemCall, значит он хочет выполнить системный вызов
                if isinstance(result, SystemCall):
                    return result

                # Если генератор вернул другой генератор —
                # значит произошёл вызов вложенной корутины
                if isinstance(result, types.GeneratorType):
                    # Сохраняем текущий генератор в стек,
                    # чтобы вернуться к нему после завершения вложенного
                    self.stack.append(self.target)

                    # Начинаем выполнение вложенного генератора
                    self.target = result
                    self.sendval = None

                else:
                    # Если результат не генератор —
                    # значит корутина просто вернула значение через yield

                    if not self.stack:
                        # Если стек пуст, то это "верхний" генератор —
                        # задача завершена
                        return

                    # Передаём результат обратно родительскому генератору
                    self.sendval = result
                    self.target = self.stack.pop()

            except StopIteration:
                # Генератор завершился (вызван return или дошёл до конца)

                if not self.stack:
                    # Если это был верхний генератор —
                    # задача полностью завершена
                    raise

                # Если это вложенный генератор —
                # возвращаемся к предыдущему
                self.sendval = None
                self.target = self.stack.pop()


### 3) Минимальный пример работы
Здесь показан небольшой сценарий с `double`, `add` и `main`, который демонстрирует, как `Task` обрабатывает вложенные генераторы и передаёт результат обратно в вызывающую корутину.

In [4]:
# Пример использования Task и SystemCall для создания простых корутин

def double(x):
  yield x * x

def add(x, y):
    yield from double(x + y)

def main():
    result = yield add(1, 2)
    print(result)
    yield

task = Task(main())
task.run()

9


### 4) Планировщик событий (`Scheduler`)
`Scheduler` управляет жизненным циклом задач и интегрируется с `selectors` для неблокирующего I/O.

Ключевые части:
- `ready` — очередь задач, готовых к исполнению;
- `task_map` — активные задачи по `tid`;
- `wait_for_read` / `wait_for_write` — регистрация ожидания файловых дескрипторов;
- `io_poll` и `io_task` — опрос событий ОС и возврат готовых задач в очередь;
- `_run_once` / `event_loop` — основной цикл кооперативного исполнения.

In [5]:
import logging
from typing import Generator, Union
from queue import Queue
from selectors import DefaultSelector, EVENT_READ, EVENT_WRITE


# Логгер для вывода информации о завершении задач
logger = logging.getLogger(__name__)


class Scheduler:
    def __init__(self):
        # Очередь задач, готовых к выполнению (FIFO)
        self.ready = Queue()
        # Селектор для отслеживания I/O-событий
        self.selector = DefaultSelector()
        # Словарь активных задач {tid: Task}
        self.task_map = {}

    def add_task(self, coroutine: Generator) -> int:
        # Создаём объект Task из генератора
        new_task = Task(coroutine)
        # Сохраняем задачу в словаре активных задач
        self.task_map[new_task.tid] = new_task
        # Добавляем задачу в очередь выполнения
        self.schedule(new_task)
        # Возвращаем id задачи
        return new_task.tid

    def exit(self, task: Task):
        # Логируем завершение задачи
        logger.info('Task %d terminated', task.tid)
        # Удаляем задачу из активных
        del self.task_map[task.tid]

    # ===== Ожидание событий ввода/вывода =====

    def wait_for_read(self, task: Task, fd: int):
        # Проверяем, зарегистрирован ли уже этот файловый дескриптор
        try:
            key = self.selector.get_key(fd)
        # Если не зарегистрирован — регистрируем его на чтение
        except KeyError:
            self.selector.register(fd, EVENT_READ, (task, None))
        # Если уже зарегистрирован — добавляем событие чтения к существующим
        else:
            mask, (reader, writer) = key.events, key.data
            self.selector.modify(fd, mask | EVENT_READ, (task, writer))

    def wait_for_write(self, task: Task, fd: int):
        # Проверяем, зарегистрирован ли fd
        try:
            key = self.selector.get_key(fd)
        # Если нет — регистрируем на запись
        except KeyError:
            self.selector.register(fd, EVENT_WRITE, (None, task))
        # Если есть — добавляем событие записи
        else:
            mask, (reader, writer) = key.events, key.data
            self.selector.modify(fd, mask | EVENT_WRITE, (reader, task))

    # ===== Удаление событий =====

    def _remove_reader(self, fd: int):
        try:
            # Получаем ключ регистрации fd в селекторе
            key = self.selector.get_key(fd)
        except KeyError:
            pass
        else:
            # Если зарегистрирован — удаляем событие чтения
            mask, (reader, writer) = key.events, key.data
            # Убираем флаг чтения
            mask &= ~EVENT_READ
            # Если событий больше нет — полностью удаляем fd
            if not mask:
                self.selector.unregister(fd)
            else:
                # Иначе обновляем регистрацию
                self.selector.modify(fd, mask, (None, writer))

    def _remove_writer(self, fd: int):
        try:
            # Получаем ключ регистрации fd в селекторе
            key = self.selector.get_key(fd)
        except KeyError:
            pass
        else:
            # Если зарегистрирован — удаляем событие записи
            mask, (reader, writer) = key.events, key.data
            # Убираем флаг записи
            mask &= ~EVENT_WRITE
            # Если событий не осталось — удаляем
            if not mask:
                self.selector.unregister(fd)
            else:
                # Иначе обновляем регистрацию
                self.selector.modify(fd, mask, (reader, None))

    # ===== Опрос I/O =====

    def io_poll(self, timeout: Union[None, float]):
        # Ждём события от ОС
        events = self.selector.select(timeout)
        # Обрабатываем каждое событие
        for key, mask in events:
            fileobj, (reader, writer) = key.fileobj, key.data
            # Если fd готов к чтению — планируем задачу
            if mask & EVENT_READ and reader is not None:
                self.schedule(reader)
                self._remove_reader(fileobj)
            # Если fd готов к записи — планируем задачу
            if mask & EVENT_WRITE and writer is not None:
                self.schedule(writer)
                self._remove_writer(fileobj)

    # ===== Фоновая задача обработки I/O =====

    def io_task(self) -> Generator:
        while True:
            # Если нет обычных задач — ждём I/O блокирующе
            if self.ready.empty():
                self.io_poll(None)
            else:
                # Если есть задачи — не блокируемся
                self.io_poll(0)
            # Передаём управление планировщику
            yield

    # ===== Планирование задач =====

    def schedule(self, task: Task):
        # Добавляем задачу в очередь ready
        self.ready.put(task)

    def _run_once(self):
        # Берём следующую задачу
        task = self.ready.get()
        try:
            # Выполняем задачу до следующей приостановки (yield)
            result = task.run()
            # Если задача приостановилась, возвращая объект SystemCall, обрабатываем его
            if isinstance(result, SystemCall):
                result.handle(self, task)
                return
        except StopIteration:
            # Если задача завершилась — удаляем её
            self.exit(task)
            return
        # Если задача ещё активна — возвращаем в очередь
        self.schedule(task)

    # ===== Главный цикл =====

    def event_loop(self):
        # Добавляем фоновую задачу обработки I/O
        self.add_task(self.io_task())
        # Пока есть активные задачи — продолжаем выполнение
        while self.task_map:
            self._run_once()

### 5) Неблокирующие операции сокета
`ReadWait` и `WriteWait` — системные вызовы, которые приостанавливают текущую задачу до готовности сокета к чтению или записи.

`AsyncSocket` — thin-wrapper над обычным `socket`, где `accept`, `recv`, `send` реализованы как корутины: перед реальным системным вызовом они отдают управление планировщику через `yield ReadWait(...)` / `yield WriteWait(...)`.

In [ ]:
from socket import socket
from typing import Tuple

class WriteWait(SystemCall):
    """
    Системный вызов для ожидания готовности к записи на файловом дескрипторе
    """
    def __init__(self, f):
        # Инициализируем системный вызов с объектом, который должен быть готов к записи (обычно это сокет)
        self.f = f

    def handle(self, sched, task):
        # Получаем файловый дескриптор из объекта сокета
        fd = self.f.fileno()
        # Регистрируем задачу на ожидание готовности к записи на этом дескрипторе
        sched.wait_for_write(task, fd)


class ReadWait(SystemCall):
    """
    Системный вызов для ожидания готовности к чтению на файловом дескрипторе
    """
    def __init__(self, f):
        # Инициализируем системный вызов с объектом, который должен быть готов к чтению (обычно это сокет)
        self.f = f

    def handle(self, sched, task):
        # Получаем файловый дескриптор из объекта сокета
        fd = self.f.fileno()
        # Регистрируем задачу на ожидание готовности к чтению на этом дескрипторе
        sched.wait_for_read(task, fd)


class AsyncSocket:
    def __init__(self, sock: socket):
        # Инициализируем асинхронный сокет, оборачивая обычный сокет
        self.sock = sock

    def accept(self) -> Tuple[AsyncSocket, Tuple[str, int]]:
        """
        Принимает новое соединение, ожидая готовности сокета к чтению.
        """
        # Принимаем новое соединение, ожидая готовности к чтению на сокете
        yield ReadWait(self.sock)
        # После того, как сокет готов к чтению, вызываем accept() для получения нового соединения
        client, addr = self.sock.accept()
        # Возвращаем новый асинхронный сокет для клиента и его адрес
        return AsyncSocket(client), addr

    def send(self, buffer: bytes):
        """
        Отправляет данные, ожидая готовности сокета к записи.
        """
        # Продолжаем отправлять данные, пока в буфере есть что отправлять
        while buffer:
            # Ждём готовности сокета к записи
            yield WriteWait(self.sock)
            # Отправляем данные и получаем количество отправленных байт
            len = self.sock.send(buffer)
            # Удаляем отправленные данные из буфера
            buffer = buffer[len:]

    def recv(self, maxbytes: int) -> bytes:
        """
        Получает данные, ожидая готовности сокета к чтению.
        """
        # Ждём готовности сокета к чтению
        yield ReadWait(self.sock)
        # Когда сокет готов к чтению, вызываем recv() для получения данных
        return self.sock.recv(maxbytes)

    def close(self):
        # Закрываем сокет, освобождая ресурсы
        yield self.sock.close()

### 6) Echo-сервер поверх собственного event loop
Финальный блок собирает все части вместе:
- `server(port)` принимает новые подключения;
- для каждого клиента создаётся отдельная задача `handle_client`;
- `handle_client` читает данные и отправляет их обратно (echo);
- `shed.event_loop()` запускает бесконечный цикл обработки задач и I/O-событий.

In [ ]:
import logging
import sys
from socket import socket, AF_INET, SOCK_STREAM

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(stream=sys.stdout))


def handle_client(client, addr):
    print("Connection from", addr)
    # Вечный цикл обработки клиента: читаем данные и отправляем их обратно, пока клиент не закроет соединение
    while True:
        # Читаем данные от клиента, ожидая готовности сокета к чтению
        data = yield from client.recv(65536)
        # Если данных нет, значит клиент закрыл соединение
        if not data:
            break
        # Отправляем данные обратно клиенту, ожидая готовности сокета к записи
        yield from client.send(data)
    print("Client closed")
    client.close()


def server(port):
    print("Server starting")
    # Создаём обычный сокет, привязываем его к порту и начинаем слушать входящие соединения
    rawsock = socket(AF_INET, SOCK_STREAM)
    rawsock.bind(("", port))
    rawsock.listen()
    # Оборачиваем обычный сокет в асинхронный, чтобы использовать его с планировщиком
    sock = AsyncSocket(rawsock)
    try:
        # Главный цикл сервера, который принимает новые соединения и планирует задачи для их обработки
        while True:
            # Принимаем новое соединение, ожидая готовности сокета к чтению
            client, addr = yield from sock.accept()
            # Планируем задачу для обработки клиента
            yield NewTask(handle_client(client, addr))
    finally:
        sock.close()


if __name__ == '__main__':
    # Создаём планировщик и добавляем задачу сервера на порт 8000
    shed = Scheduler()
    shed.add_task(server(8000))

    try:
        # Запускаем главный цикл планировщика, который будет обрабатывать задачи и I/O события
        shed.event_loop()
    except KeyboardInterrupt:
        print("\nСервер остановлен пользователем")

Server starting
Connection from ('127.0.0.1', 57776)
Client closed
Task 4 terminated

Сервер остановлен пользователем
